# Week 5 Curriculum Hybrid RAG (OpenRouter)

Notebook runner for the community contribution project.

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv


def locate_project_dir() -> Path:
    """Find the project dir robustly from different notebook launch locations."""
    target_rel = Path("week5/community-contributions/BernardUdo/week5-curriculum-hybrid-rag")
    cwd = Path.cwd().resolve()

    for base in [cwd, *cwd.parents]:
        direct = (base / "curriculum_hybrid_rag.py").resolve()
        if direct.exists():
            return direct.parent

        nested = (base / target_rel / "curriculum_hybrid_rag.py").resolve()
        if nested.exists():
            return nested.parent

    raise FileNotFoundError("Could not locate week5-curriculum-hybrid-rag/curriculum_hybrid_rag.py")


PROJECT_DIR = locate_project_dir()
REPO_ROOT = next((p for p in [PROJECT_DIR, *PROJECT_DIR.parents] if (p / ".git").exists()), PROJECT_DIR)
SOURCE_DIR = REPO_ROOT / "week5/knowledge-base"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"

load_dotenv(override=True)
assert os.getenv("OPENROUTER_API_KEY"), "OPENROUTER_API_KEY missing in .env"
assert SOURCE_DIR.exists(), f"Knowledge base folder not found: {SOURCE_DIR}"

print("Environment loaded.")
print(f"Project: {PROJECT_DIR}")
print(f"Knowledge base: {SOURCE_DIR}")

In [ ]:
import importlib
import sys

project_path = str(PROJECT_DIR.resolve())
if project_path not in sys.path:
    sys.path.insert(0, project_path)

module = importlib.import_module("curriculum_hybrid_rag")
module = importlib.reload(module)

assistant = module.HybridRagAssistant(artifacts_dir=ARTIFACTS_DIR)
print(f"Module loaded from: {PROJECT_DIR / 'curriculum_hybrid_rag.py'}")

In [ ]:
# Optional: rebuild index
# Keep False by default to avoid unnecessary re-embedding/API usage on every run.
import shutil

rebuild_index = False

if rebuild_index:
    if ARTIFACTS_DIR.exists():
        shutil.rmtree(ARTIFACTS_DIR)
        print(f"Deleted artifacts at: {ARTIFACTS_DIR}")
    assistant.build_index(source_dir=SOURCE_DIR)
    print(f"Fresh index created with {len(assistant.chunks)} chunks at {ARTIFACTS_DIR}")
else:
    print("Rebuild skipped. Use Cell 4 to load existing index or build if missing.")

In [ ]:
# Build or refresh index (safe mode)
if "assistant" not in globals():
    raise RuntimeError("Run the module-loading cell first (Cell 2) to initialize `assistant`.")

chunks_path = ARTIFACTS_DIR / "chunks.json"
embeddings_path = ARTIFACTS_DIR / "embeddings.npy"

if chunks_path.exists() and embeddings_path.exists():
    assistant.load_index()
    print(f"Loaded existing index with {len(assistant.chunks)} chunks from {ARTIFACTS_DIR}")
else:
    assistant.build_index(source_dir=SOURCE_DIR)
    print(f"Indexed {len(assistant.chunks)} chunks into {ARTIFACTS_DIR}")

In [ ]:
import json
import re
from urllib.parse import quote
from urllib.request import urlopen


def _normalize_web_query(query: str) -> str:
    """Lightly simplify question text for better public search matching."""
    cleaned = query.strip()
    cleaned = re.sub(r"\s+", " ", cleaned)
    cleaned = re.sub(r"\baccording to .*", "", cleaned, flags=re.IGNORECASE)
    return cleaned.strip(" ?")


def fetch_online_references(query: str, limit: int = 3) -> list[tuple[str, str]]:
    """Fetch lightweight web references from Wikipedia OpenSearch (no API key needed)."""
    normalized = _normalize_web_query(query)
    candidates = [query]
    if normalized and normalized.lower() != query.lower():
        candidates.append(normalized)

    for q in candidates:
        url = (
            "https://en.wikipedia.org/w/api.php"
            f"?action=opensearch&search={quote(q)}&limit={limit}&namespace=0&format=json"
        )
        try:
            with urlopen(url, timeout=8) as resp:
                payload = json.loads(resp.read().decode("utf-8"))
            titles = payload[1] if len(payload) > 1 else []
            links = payload[3] if len(payload) > 3 else []
            refs = list(zip(titles, links))
            if refs:
                return refs
        except Exception:
            continue
    return []


def web_search_fallback_links(query: str) -> list[str]:
    q = quote(_normalize_web_query(query) or query)
    return [
        f"https://www.google.com/search?q={q}",
        f"https://duckduckgo.com/?q={q}",
    ]


# Change only this line for each test (keep this as a single-line string)
question = "What is Insurellm core mission according to company docs?"

# Set True when you want external reference links in output.
include_online_references = True

# assistant.answer() now contains built-in graceful fallback for 402/API credit errors.
# It returns extractive local answer + fallback audit instead of raising.
answer_text, hits, report = assistant.answer(question)

print("QUESTION:", question)
print("\nANSWER:\n", answer_text)

print("\nSOURCES (LOCAL RAG):")
for h in hits:
    print(f"- {h.chunk.source}#{h.chunk.chunk_id} (score={h.score:.3f})")

if include_online_references:
    refs = fetch_online_references(question, limit=3)
    print("\nONLINE REFERENCES:")
    if refs:
        print("- Wikipedia matches:")
        for title, link in refs:
            print(f"  - {title}: {link}")
    else:
        print("- No direct Wikipedia matches found.")
        print("- Try search engine links:")
        for link in web_search_fallback_links(question):
            print(f"  - {link}")

print("\nAUDIT:")
print(
    f"groundedness={report.groundedness}, "
    f"completeness={report.completeness}, "
    f"citation_quality={report.citation_quality}"
)
print("feedback:", report.feedback)